# DWD-Wetterdaten Berlin ab 2020

Dieses Notebook lädt ausschließlich tägliche DWD-Messwerte als CSV herunter. Es führt noch keine Wetteranalyse durch.

**Station:** Berlin-Dahlem (FU), DWD-ID `00403`, nahe Berlin-Zentrum. **Quelle:** DWD Climate Data Center, Tageswerte (KL). Der Download kombiniert die historische Datei bis 2025-12-31 mit der aktuellen Datei ab 2026 und reicht bis zum jüngsten dort verfügbaren Messdatum.

## Verfügbarkeit der Messgrößen

Die Stationsmetadaten weisen alle angeforderten Messgrößen bis zum **01.11.2026** aus: `TNK` (Tagesminimum, °C), `TXK` (Tagesmaximum, °C), `TMK` (Tagesmittel, °C), `RSK` (Niederschlagssumme, mm) und `SDK` (Sonnenscheindauer, Stunden).

Im Zeitraum 2020-01-01 bis 2026-11-01 liegen 2.483 Tagesdatensätze für 2.497 Kalendertage vor. Vom **04.03. bis 17.03.2025** fehlt ein kompletter Datensatz; außerdem gibt es vereinzelt fehlende Einzelwerte. Der Export ergänzt alle Kalendertage und schreibt fehlende Messwerte als leere CSV-Felder. Damit sind die Daten verfügbar, aber nicht lückenlos.

### Spalten und Quellen

| CSV-Spalte | DWD-Feld | Bedeutung | Einheit |
| --- | --- | --- | --- |
| `temp_min_c` | `TNK` | Tagesminimum in 2 m Höhe | °C |
| `temp_max_c` | `TXK` | Tagesmaximum in 2 m Höhe | °C |
| `temp_durchschnitt_c` | `TMK` | Tagesmitteltemperatur | °C |
| `niederschlag_mm` | `RSK` | tägliche Niederschlagssumme | mm |
| `sonnenstunden_h` | `SDK` | tägliche Sonnenscheindauer | h |

Historische Daten: [DWD-Archiv Berlin-Dahlem](https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/). Aktuelle Daten: [DWD-Recent Berlin-Dahlem](https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/recent/). Der DWD-Kennwert `-999` wird als fehlender Wert exportiert.

In [1]:
import csv
import io
import urllib.request
import zipfile
from datetime import date, datetime, timedelta
from pathlib import Path

## CSV herunterladen

Die folgende Zelle lädt die DWD-Historie und die aktuelle Tagesdatei, kombiniert sie ohne doppelte Tage und speichert eine CSV im aktuellen Notebook-Arbeitsverzeichnis. Für jeden Kalendertag ab 2020-01-01 gibt es eine Zeile; fehlende Messungen bleiben leer.

In [2]:
dwd_base_url = (
    "https://opendata.dwd.de/climate_environment/CDC/observations_germany/"
    "climate/daily/kl/"
 )
historical_url = (
    dwd_base_url
    + "historical/tageswerte_KL_00403_19500101_20251231_hist.zip"
 )
recent_url = dwd_base_url + "recent/tageswerte_KL_00403_akt.zip"
start_date = date(2020, 1, 1)
today = date.today()

def read_dwd_product(url):
    with urllib.request.urlopen(url, timeout=60) as response:
        archive_bytes = response.read()

    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        product_name = next(
            name
            for name in archive.namelist()
            if name.startswith("produkt_klima_tag_") and name.endswith(".txt")
        )
        product_text = archive.read(product_name).decode("latin-1")

    reader = csv.DictReader(io.StringIO(product_text), delimiter=";")
    return [
        {
            key.strip(): (value or "").strip()
            for key, value in row.items()
            if key
        }
        for row in reader
    ]

def get_measurement_date(row):
    value = row.get("MESS_DATUM", "")
    try:
        return datetime.strptime(value, "%Y%m%d").date()
    except ValueError:
        return None

historical_rows = read_dwd_product(historical_url)
recent_rows = read_dwd_product(recent_url)
historical_by_date = {}

for row in historical_rows:
    measurement_date = get_measurement_date(row)
    if measurement_date is not None and start_date <= measurement_date <= today:
        historical_by_date[measurement_date] = row

if not historical_by_date:
    raise ValueError("Die DWD-Historie enthält keine Daten ab 2020.")

historical_end = max(historical_by_date)
records_by_date = historical_by_date.copy()

for row in recent_rows:
    measurement_date = get_measurement_date(row)
    if (
        measurement_date is not None
        and historical_end < measurement_date <= today
    ):
        records_by_date[measurement_date] = row

latest_date = max(records_by_date)
if latest_date < start_date:
    raise ValueError("Die DWD-Dateien enthalten keine Daten ab 2020.")

column_map = {
    "temp_min_c": "TNK",
    "temp_max_c": "TXK",
    "temp_durchschnitt_c": "TMK",
    "niederschlag_mm": "RSK",
    "sonnenstunden_h": "SDK",
}
csv_columns = ["datum", *column_map.keys()]
csv_rows = []
missing_record_dates = []
current_date = start_date

while current_date <= latest_date:
    source_row = records_by_date.get(current_date)
    if source_row is None:
        missing_record_dates.append(current_date)

    output_row = {"datum": current_date.isoformat()}
    for output_column, dwd_column in column_map.items():
        value = source_row.get(dwd_column, "") if source_row else ""
        try:
            is_missing = not value or float(value) == -999
        except ValueError:
            is_missing = True
        output_row[output_column] = "" if is_missing else value

    csv_rows.append(output_row)
    current_date += timedelta(days=1)

csv_path = Path.cwd() / "dwd_berlin_dahlem_2020_bis_heute.csv"
with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=csv_columns)
    writer.writeheader()
    writer.writerows(csv_rows)

print("Station: Berlin-Dahlem (FU), DWD-ID 00403")
print(f"Zeitraum: {start_date} bis {latest_date}")
print(f"Tageszeilen: {len(csv_rows)}; fehlende komplette Tage: {len(missing_record_dates)}")
if missing_record_dates:
    print(
        "Lückenzeitraum: "
        f"{min(missing_record_dates)} bis {max(missing_record_dates)}"
    )
for output_column in column_map:
    missing_count = sum(not row[output_column] for row in csv_rows)
    print(f"Fehlende Werte {output_column}: {missing_count}")
print(f"CSV gespeichert unter: {csv_path.resolve()}")

Station: Berlin-Dahlem (FU), DWD-ID 00403
Zeitraum: 2020-01-01 bis 2026-11-01
Tageszeilen: 2497; fehlende komplette Tage: 14
Lückenzeitraum: 2025-03-04 bis 2025-03-17
Fehlende Werte temp_min_c: 16
Fehlende Werte temp_max_c: 16
Fehlende Werte temp_durchschnitt_c: 16
Fehlende Werte niederschlag_mm: 15
Fehlende Werte sonnenstunden_h: 16
CSV gespeichert unter: /workspaces/seminar3/dwd_berlin_dahlem_2020_bis_heute.csv
